# Understanding the Constant-Q Transform (CQT)

This notebook breaks down exactly how the Tab-estimator project processes audio into features using the Constant-Q Transform. 

The CQT is especially useful for music because its frequency bins are logarithmically spaced, meaning it aligns perfectly with human pitch perception and the frets on a guitar. Let's go through the implementation step by step.

In [2]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import IPython.display as ipd

ModuleNotFoundError: No module named 'librosa'

## 1. Load the Audio File

First, we load the audio file. In `config.yaml` of the Tab-estimator project, the `down_sampling_rate` is set to `22050`. We'll use this same sample rate here.

In [ ]:
audio_path = '00_BN1-129-Eb_comp_mic.wav'

# Load the audio time series (y) and sampling rate (sr)
y, sr = librosa.load(audio_path, sr=22050)

print("Audio loaded!")
print(f"Sample rate: {sr} Hz")
print(f"Total samples: {len(y)}")
print(f"Duration: {len(y)/sr:.2f} seconds")

# This creates a playable audio widget right here in the notebook!
ipd.Audio(y, rate=sr)

## 2. Compute the CQT

Now we apply the Constant-Q Transform. We'll use the exact parameters from the project's `config.yaml`:
- `hop_length = 512`: The number of audio samples between each frame (column) in the CQT window.
- `bins_per_octave = 24`: The pitch resolution. Since a standard Western octave has 12 semitones, 24 means we have **quarter-tone resolution** (more precise than standard notes).
- `n_bins = 192`: The total number of frequency bins to track. 192 bins / 24 bins per octave = exactly 8 octaves of range.

In [ ]:
hop_length = 512
bins_per_octave = 24
n_bins = 192

# Compute CQT 
# This returns complex numbers containing both magnitude (loudness) and phase
C = librosa.cqt(y, sr=sr, 
                hop_length=hop_length, 
                n_bins=n_bins, 
                bins_per_octave=bins_per_octave)

# For our neural network, we only care about the magnitude (loudness of the frequencies), 
# so we take the absolute value.
C_mag = np.abs(C)

print(f"CQT Magnitude Shape: {C_mag.shape}")
print(f"(Frequency Bins: {C_mag.shape[0]}, Time Frames: {C_mag.shape[1]})")

## 3. Convert to Decibels

Human ears perceive loudness logarithmically, not linearly. To make our data visually and computationally meaningful for the neural network, we compress the large amplitude ranges into a decibel (dB) scale.

In [ ]:
# Convert amplitude (magnitude) to decibels
C_db = librosa.amplitude_to_db(C_mag, ref=np.max)

print("Converted to Decibel scale.")
print(f"Max value (dB): {np.max(C_db):.2f}")
print(f"Min value (dB): {np.min(C_db):.2f}")

## 4. Visualize the Result

Finally, let's plot the CQT spectrogram. This gives us a 2D image where:
- The **X-axis** is time (seconds)
- The **Y-axis** is pitch (frequency in Hz, spaced logarithmically)
- The **Color** shows the intensity (loudness) of the note

This dense 2D image is exactly what gets fed into the machine learning model as the "input feature" instead of the raw audio waveform.

In [ ]:
plt.figure(figsize=(16, 6))

# specshow automatically formats the axes for music data based on our parameters
librosa.display.specshow(C_db, sr=sr, hop_length=hop_length, 
                         x_axis='time', y_axis='cqt_hz', 
                         bins_per_octave=bins_per_octave, 
                         cmap='magma') # Try 'magma', 'viridis', or 'coolwarm'

plt.colorbar(format='%+2.0f dB')
plt.title('Constant-Q Transform (CQT) Spectrogram - Guitar Recording')
plt.tight_layout()
plt.show()